# MNIST MLP3 — AdamW baseline

AdamW is applied to every trainable parameter with its own baseline learning rate and decoupled weight decay.

This is a clean optimizer baseline: no trace-log projection, adaptive ECS correction,
WW-PGD, or spectral-flow intervention. Every epoch, including epoch 0, measures
full train/test loss and accuracy plus the original WeightWatcher full-$M$
diagnostics.

All artifacts from the three baseline notebooks use one shared output root. By
default that is `baseline/runs/` inside the checked-out repository. Set the
environment variable `RG_BASELINE_OUTPUT_ROOT` before running the notebook to
redirect all three runs to another directory, for example
`/tmp/rg_optimizers_baselines`.

This notebook saves:

- one checkpoint per trained epoch in `<run>/checkpoints/epoch_XXX.pt`;
- the final model and optimizer state in `<run>/final_state.pt`;
- CSV metric histories, the ESD history, configuration, and plots in `<run>/`.

$$
m_{\mathrm{mid}}=\left\lfloor
\frac{m_{\mathrm{detX}}+m_{\mathrm{PL}}}{2}
\right\rfloor.
$$

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "weightwatcher>=0.7.7",
    ])

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    baseline_candidate = candidate / "baseline"
    if (baseline_candidate / "rg_baselines").is_dir():
        ROOT = baseline_candidate
        break
    if (candidate / "rg_baselines").is_dir():
        ROOT = candidate
        break

if ROOT is None:
    raise RuntimeError("Run from a clone of CalculatedContent/rg_optimizers")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

output_override = os.environ.get("RG_BASELINE_OUTPUT_ROOT")
OUTPUT_ROOT = (
    Path(output_override).expanduser().resolve()
    if output_override
    else (ROOT / "runs").resolve()
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("baseline root:", ROOT.resolve())
print("shared output root:", OUTPUT_ROOT)

In [ ]:
from rg_baselines import BaselineConfig, plot_all, run_baseline
from IPython.display import display

CONFIG = BaselineConfig(
    optimizer="adamw",
    epochs=20,
    train_eval_max_batches=None,
    adamw_learning_rate=1e-3,
    adamw_weight_decay=1e-2,
    save_epoch_checkpoints=True,
)
RUN_DIR = OUTPUT_ROOT / CONFIG.run_slug

result = run_baseline(
    CONFIG,
    data_dir=ROOT / "data",
    output_dir=RUN_DIR,
    progress=True,
)
plot_all(result, output_dir=RUN_DIR / "plots", show=True)

expected_checkpoints = [
    RUN_DIR / "checkpoints" / f"epoch_{epoch:03d}.pt"
    for epoch in range(1, CONFIG.epochs + 1)
]
missing_checkpoints = [
    path for path in expected_checkpoints if not path.is_file()
]
if missing_checkpoints:
    raise RuntimeError(
        "Missing epoch checkpoints: "
        + ", ".join(str(path) for path in missing_checkpoints)
    )
if not (RUN_DIR / "final_state.pt").is_file():
    raise RuntimeError(f"Missing final state: {RUN_DIR / 'final_state.pt'}")

print("saved run:", RUN_DIR.resolve())
print("epoch checkpoints:", len(expected_checkpoints))
print("final state:", (RUN_DIR / "final_state.pt").resolve())

In [ ]:
import numpy as np

performance_for_display = result.performance.copy()
performance_for_display["train_perplexity"] = np.exp(
    performance_for_display["train_loss"]
)
performance_for_display["test_perplexity"] = np.exp(
    performance_for_display["test_loss"]
)
display(performance_for_display)

required = [
    "run",
    "epoch",
    "layer",
    "alpha",
    "detX_num",
    "num_pl_spikes",
    "ERG_gap",
    "m_midpoint",
    "trace_log_midpoint_total",
    "trace_log_midpoint_per_eval",
    "geometric_mean_midpoint",
    "stable_rank",
    "participation_ratio",
    "midpoint_energy_fraction",
    "status",
]
display(result.spectral_metrics[required].sort_values(["epoch", "layer"]))
display(result.optimizer_groups)